# Session 02 — Autoencoders & Variational Autoencoders**Dataset:** Fashion-MNIST  **Frameworks:** PyTorch, matplotlib

## Objectives1. Load and explore Fashion-MNIST.2. Build and train a standard Autoencoder with **MSE** loss.3. Build and train a standard Autoencoder with **BCE** loss.4. Compare reconstructions from both loss functions.5. Implement a **Denoising Autoencoder**.6. Build and train a **Variational Autoencoder (VAE)**.7. Sample from the latent space to generate new images.8. Plot all loss curves.

## Theory### AutoencoderAn autoencoder learns to compress input $x$ into a low-dimensional latent code $z = f_\theta(x)$ and reconstruct it as $\hat{x} = g_\phi(z)$.**MSE Loss:** $\mathcal{L} = \frac{1}{n}\sum(x - \hat{x})^2$ — treats pixel errors uniformly.**BCE Loss:** $\mathcal{L} = -\sum[x \log \hat{x} + (1-x)\log(1-\hat{x})]$ — better for binary/normalized images.### Variational AutoencoderA VAE adds a probabilistic twist: the encoder outputs $\mu$ and $\log\sigma^2$ that parametrise a Gaussian. The **reparameterisation trick** $z = \mu + \sigma \cdot \epsilon$ enables back-prop.**ELBO Loss:** $\mathcal{L} = \text{Reconstruction} + D_{KL}(q(z|x) \| p(z))$```mermaidgraph LR    X[Input x] --> E[Encoder]    E --> MU[μ]    E --> LV[log σ²]    MU --> RP[Reparameterize]    LV --> RP    RP --> Z[z]    Z --> D[Decoder]    D --> XH[x̂]    style Z fill:#f9f,stroke:#333```

## Learning Outcomes- Implement encoder-decoder architectures in PyTorch.- Understand the effect of loss function choice on reconstruction quality.- Apply noise augmentation for denoising autoencoders.- Sample from a learned latent space using the reparameterisation trick.- Interpret loss curves and reconstruction quality.

---## Setup

In [ ]:
import sysfrom pathlib import PathPROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()if str(PROJECT_ROOT) not in sys.path:    sys.path.insert(0, str(PROJECT_ROOT))import torchimport matplotlib.pyplot as pltimport numpy as npfrom src.utils import seed_everything, get_device, setup_loggingfrom src.config import GENERATED_IMAGES_DIR, ensure_dirsfrom src.data_loader import get_fashion_mnist_loaders, add_noise, FASHION_MNIST_CLASSESfrom src.autoencoder import Autoencoder, VAE, train_autoencoder, train_vaefrom src.visualization import (    plot_loss_curves, plot_image_grid,    plot_reconstruction_comparison, save_generated_images)setup_logging()seed_everything(42)ensure_dirs()device = get_device()print(f'Device: {device}')

---## 1. Load Fashion-MNIST

In [ ]:
train_loader, test_loader = get_fashion_mnist_loaders(
    batch_size=64, flatten=True
)  # Peek at a batchsample_batch, sample_labels = next(iter(test_loader))print(f'Batch shape: {sample_batch.shape}')print(f'Label examples: {[FASHION_MNIST_CLASSES[l] for l in sample_labels[:5].tolist()]}')

In [ ]:
# Visualise samples (unflatten for display)plot_image_grid(sample_batch[:16].view(-1, 1, 28, 28), nrow=8,                title='Fashion-MNIST Samples')

---## 2. Autoencoder with MSE Loss

In [ ]:
ae_mse = Autoencoder(latent_dim=16, loss_function='mse')print(f'AE (MSE) parameters: {sum(p.numel() for p in ae_mse.parameters()):,}')mse_losses = train_autoencoder(ae_mse, train_loader, device, epochs=10, lr=1e-3)

In [ ]:
plot_loss_curves(mse_losses, title="Autoencoder — MSE Loss")

In [ ]:
# Reconstructionsae_mse.eval()with torch.no_grad():    test_batch = sample_batch[:8].to(device)    recon_mse = ae_mse(test_batch)plot_reconstruction_comparison(    test_batch.view(-1, 1, 28, 28),    recon_mse.view(-1, 1, 28, 28),    title='MSE Autoencoder Reconstruction')

---## 3. Autoencoder with BCE Loss

In [ ]:
ae_bce = Autoencoder(latent_dim=16, loss_function='bce')bce_losses = train_autoencoder(ae_bce, train_loader, device, epochs=10, lr=1e-3)

In [ ]:
plot_loss_curves(bce_losses, title="Autoencoder — BCE Loss")

In [ ]:
# Reconstructionsae_bce.eval()with torch.no_grad():    recon_bce = ae_bce(test_batch)plot_reconstruction_comparison(    test_batch.view(-1, 1, 28, 28),    recon_bce.view(-1, 1, 28, 28),    title='BCE Autoencoder Reconstruction')

---## 4. Reconstruction Comparison — MSE vs BCE

In [ ]:
fig, axes = plt.subplots(3, 8, figsize=(14, 5))for i in range(8):    axes[0, i].imshow(test_batch[i].cpu().view(28, 28), cmap='gray')    axes[0, i].axis('off')    axes[1, i].imshow(recon_mse[i].cpu().view(28, 28), cmap='gray')    axes[1, i].axis('off')    axes[2, i].imshow(recon_bce[i].cpu().view(28, 28), cmap='gray')    axes[2, i].axis('off')axes[0, 0].set_title('Original', fontsize=10)axes[1, 0].set_title('MSE', fontsize=10)axes[2, 0].set_title('BCE', fontsize=10)fig.suptitle('Reconstruction Comparison: MSE vs BCE', fontsize=14)fig.tight_layout()plt.show()

### Interpretation- **MSE** minimises pixel-level squared error and tends to produce smoother/blurrier results.- **BCE** treats each pixel as a Bernoulli variable and often preserves sharper edges on binary-like images.- For Fashion-MNIST (normalised to [0,1]), BCE generally gives crisper reconstructions.

---## 5. Denoising Autoencoder

In [ ]:
# Train a denoising AE: input noisy, target cleanae_denoise = Autoencoder(latent_dim=16, loss_function='mse')ae_denoise.to(device)optimizer = torch.optim.Adam(ae_denoise.parameters(), lr=1e-3)denoise_losses = []for epoch in range(1, 11):    ae_denoise.train()    epoch_loss = 0.0    for images, _ in train_loader:        clean = images.to(device)        noisy = add_noise(clean, noise_factor=0.5)        recon = ae_denoise(noisy)        loss = ae_denoise.compute_loss(clean, recon)        optimizer.zero_grad()        loss.backward()        optimizer.step()        epoch_loss += loss.item() * clean.size(0)    avg = epoch_loss / len(train_loader.dataset)    denoise_losses.append(avg)    print(f'Epoch {epoch}/10 | Denoise Loss: {avg:.6f}')plot_loss_curves(denoise_losses, title='Denoising Autoencoder Loss')

In [ ]:
# Show denoising resultsae_denoise.eval()noisy_test = add_noise(test_batch, noise_factor=0.5)with torch.no_grad():    denoised = ae_denoise(noisy_test)fig, axes = plt.subplots(3, 8, figsize=(14, 5))for i in range(8):    axes[0, i].imshow(test_batch[i].cpu().view(28, 28), cmap='gray')    axes[0, i].axis('off')    axes[1, i].imshow(noisy_test[i].cpu().view(28, 28), cmap='gray')    axes[1, i].axis('off')    axes[2, i].imshow(denoised[i].cpu().view(28, 28), cmap='gray')    axes[2, i].axis('off')axes[0, 0].set_title('Clean', fontsize=10)axes[1, 0].set_title('Noisy', fontsize=10)axes[2, 0].set_title('Denoised', fontsize=10)fig.suptitle('Denoising Autoencoder Results', fontsize=14)fig.tight_layout()plt.show()

---## 6. Variational Autoencoder (VAE)

In [ ]:
vae = VAE(latent_dim=16)print(f'VAE parameters: {sum(p.numel() for p in vae.parameters()):,}')vae_losses = train_vae(vae, train_loader, device, epochs=10, lr=1e-3)

In [ ]:
plot_loss_curves(vae_losses, title="VAE ELBO Loss")

In [ ]:
# VAE Reconstructionsvae.eval()with torch.no_grad():    recon_vae, _, _ = vae(test_batch.to(device))plot_reconstruction_comparison(    test_batch.view(-1, 1, 28, 28),    recon_vae.view(-1, 1, 28, 28),    title='VAE Reconstruction')

---## 7. Latent Sampling — Generate New Images

In [ ]:
# Sample 5 images from the VAE latent spacevae.eval()with torch.no_grad():    generated = vae.sample(5, device)plot_image_grid(generated, nrow=5, title='VAE Generated Images (5 samples)')# Save generated imagessaved_paths = save_generated_images(generated, prefix='vae_generated')print(f'Saved {len(saved_paths)} images to {GENERATED_IMAGES_DIR}')

In [ ]:
# Generate a larger grid for visual appealwith torch.no_grad():    generated_16 = vae.sample(16, device)plot_image_grid(generated_16, nrow=8, title='VAE — 16 Generated Samples')

---## 8. All Loss Curves Combined

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))epochs = range(1, 11)ax.plot(epochs, mse_losses, 'o-', label='AE (MSE)', linewidth=2)ax.plot(epochs, bce_losses, 's-', label='AE (BCE)', linewidth=2)ax.plot(epochs, denoise_losses, '^-', label='Denoising AE', linewidth=2)ax.plot(epochs, vae_losses, 'D-', label='VAE (ELBO)', linewidth=2)ax.set_xlabel('Epoch')ax.set_ylabel('Loss')ax.set_title('Training Loss Comparison — All Models')ax.legend()ax.grid(True, alpha=0.3)fig.tight_layout()plt.show()

### Interpretation- MSE and BCE losses converge rapidly for standard autoencoders.- The denoising AE shows higher initial loss (noisy inputs) but learns robust features.- VAE ELBO loss is on a different scale because it includes the KL-divergence term.- All models converge within 10 epochs on Fashion-MNIST.

---## Summary| Model | Loss | Key Feature ||-------|------|------------|| AE (MSE) | Mean Squared Error | Pixel-level reconstruction || AE (BCE) | Binary Cross-Entropy | Sharper edges on binary images || Denoising AE | MSE on clean targets | Learns robust, noise-invariant features || VAE | ELBO (BCE + KLD) | Smooth latent space, generative sampling |

---## Interview Questions1. Why does an autoencoder with too-large a latent dimension memorise instead of generalise?2. Explain the reparameterisation trick. Why is it necessary?3. What happens if you set the KL weight to zero in a VAE?4. Compare MSE vs BCE loss for image reconstruction tasks.5. How does a denoising autoencoder differ from a standard autoencoder in terms of learned representations?6. What is posterior collapse in VAEs and how can it be prevented?7. How would you evaluate generation quality beyond visual inspection?8. Can autoencoders be used for anomaly detection? Explain.

---## References1. Kingma, D. P. & Welling, M. (2014). *Auto-Encoding Variational Bayes*. ICLR.2. Vincent, P. et al. (2008). *Extracting and Composing Robust Features with Denoising Autoencoders*. ICML.3. Xiao, H., Rasul, K. & Vollgraf, R. (2017). *Fashion-MNIST: a Novel Image Dataset for Benchmarking ML Algorithms*.4. Doersch, C. (2016). *Tutorial on Variational Autoencoders*. arXiv:1606.05908.